# Ablation Studies

1. **Frozen vs. Fine-tuned embeddings** — compare Ridge (frozen AbLang2) vs. LoRA (fine-tuned AbLang2)
2. **CDR3-only vs. Full-sequence** — mask non-CDR3 regions and measure impact on LoRA fine-tuning

In [ ]:
import os, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

import ablang2
from peft import LoraConfig, TaskType, get_peft_model

OUTPUT_DIR = "results/ablations"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Using device: {DEVICE}")

In [ ]:
# Shared constants (must match main notebooks)
GDPA1_DATA_PATH = "GDPa1_v1.2_20250814.csv"
VH_COL = "vh_protein_sequence"
VL_COL = "vl_protein_sequence"
FOLD_COL = "hierarchical_cluster_IgG_isotype_stratified_fold"
LABEL_COL = "label"

REGRESSION_TARGETS = [
    "HIC", "SMAC", "HAC", "PR_Ova", "PR_CHO",
    "SEC_%Monomer", "AC-SINS_pH6.0", "AC-SINS_pH7.4",
    "Tm1", "Tm2", "Titer",
]

LORA_R = 8; LORA_ALPHA = 16; LORA_DROPOUT = 0.1
EPOCHS = 30; BATCH_SIZE = 8; LR = 1e-4; WEIGHT_DECAY = 1e-2; HEAD_DROPOUT = 0.1

In [ ]:
df = pd.read_csv(GDPA1_DATA_PATH)

approved_mask = df["highest_clinical_trial_asof_feb2025"] == "Approved"
not_approved_mask = (df["est_status_asof_feb2025"] == "Discontinued") & ~approved_mask
df_model = df[approved_mask | not_approved_mask].copy()
df_model["label"] = approved_mask[approved_mask | not_approved_mask].astype(int)

print(f"Antibodies for modeling: {len(df_model)}")

## Ablation 1: Frozen vs. Fine-tuned Embeddings

Compile from existing result files — no re-training needed.

In [ ]:
from pathlib import Path

lora_approval = pd.read_csv("results/ablang2_developability/ablang2_fine_tune_LoRA_approval_cv_results.csv")
ridge_approval = pd.read_csv("results/ablang2_ridge/ablang2_ridge_approval_cv_results.csv")

lora_reg = pd.read_csv("results/ablang2_developability/regression_cv_results.csv")
ridge_reg = pd.read_csv("results/ablang2_ridge/ablang2_ridge_regression_cv_results.csv")

# Approval ablation
ablation_approval = pd.DataFrame({
    "Condition": ["Fine-tuned (LoRA)", "Frozen + Ridge"],
    "AUPRC_mean": [lora_approval["AUPRC"].mean(), ridge_approval["AUPRC"].mean()],
    "AUPRC_std":  [lora_approval["AUPRC"].std(),  ridge_approval["AUPRC"].std()],
    "AUROC_mean": [lora_approval["AUROC"].mean(), ridge_approval["AUROC"].mean()],
    "AUROC_std":  [lora_approval["AUROC"].std(),  ridge_approval["AUROC"].std()],
}).round(3)

print("Ablation 1a: Frozen vs. Fine-tuned — Approval Prediction")
print(ablation_approval.to_string(index=False))

# Regression ablation
def _mean_rho(df):
    return df.groupby("target")["spearman_rho"].mean()

lora_means = _mean_rho(lora_reg)
ridge_means = _mean_rho(ridge_reg)

ablation_reg = pd.DataFrame({
    "target": lora_means.index,
    "LoRA_rho": lora_means.values,
    "Frozen_Ridge_rho": ridge_means.reindex(lora_means.index).values,
}).round(3).sort_values("LoRA_rho", ascending=False)
ablation_reg["delta"] = (ablation_reg["LoRA_rho"] - ablation_reg["Frozen_Ridge_rho"]).round(3)

print("\nAblation 1b: Frozen vs. Fine-tuned — Regression (mean ρ)")
print(ablation_reg.to_string(index=False))

ablation_approval.to_csv(f"{OUTPUT_DIR}/ablation_frozen_vs_finetuned_approval.csv", index=False)
ablation_reg.to_csv(f"{OUTPUT_DIR}/ablation_frozen_vs_finetuned_regression.csv", index=False)

## Ablation 2: CDR3-only vs. Full-sequence LoRA

Replace non-CDR3 residues with a mask character ('X') so AbLang2 only sees CDR3 signal. Compare approval AUPRC/AUROC under 5-fold CV.

In [ ]:
# CDR3 extraction heuristic using IMGT-like patterns
# For VH: conserved C...W/F motif at CDR-H3 boundaries
# For VL: conserved C...F motif at CDR-L3 boundaries
# Falls back to last 15 residues if pattern not found

def extract_cdr3_mask(seq, chain="H"):
    """Replace non-CDR3 residues with 'X'. Keep CDR3 intact."""
    seq = str(seq).upper().strip()
    if chain == "H":
        # CDR-H3: between conserved Cys (pos ~104) and Trp/Phe-Gly (pos ~118)
        match = re.search(r'(C)([A-Z]{3,30})(W[GS])', seq)
    else:
        # CDR-L3: between conserved Cys and Phe-Gly
        match = re.search(r'(C)([A-Z]{3,20})(F[GS])', seq)

    if match:
        start = match.start()
        end = match.end()
        masked = "X" * start + seq[start:end] + "X" * (len(seq) - end)
        return masked
    else:
        # Fallback: keep last 15 residues as approximate CDR3
        k = min(15, len(seq))
        return "X" * (len(seq) - k) + seq[-k:]


# Test
sample_vh = df_model[VH_COL].iloc[0]
print(f"Original VH (first 80): {sample_vh[:80]}")
print(f"CDR3-masked (first 80): {extract_cdr3_mask(sample_vh, 'H')[:80]}")

In [ ]:
# Reuse model classes from main notebook
# (copy minimal definitions to keep this self-contained)

class AbLang2LoRAEncoder(nn.Module):
    LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]

    def __init__(self, ablang2_model, lora_r=LORA_R, lora_alpha=LORA_ALPHA,
                 lora_dropout=LORA_DROPOUT, freeze_non_lora=True):
        super().__init__()
        self.ablang = ablang2_model
        self.tokenizer = ablang2_model.tokenizer
        self.padding_tkn = ablang2_model.AbLang.AbRep.padding_tkn
        lora_cfg = LoraConfig(
            r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
            target_modules=self.LORA_TARGET_MODULES, bias="none",
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        self.ablang.AbLang = get_peft_model(self.ablang.AbLang, lora_cfg)
        if freeze_non_lora:
            for name, param in self.ablang.AbLang.named_parameters():
                if "lora_" not in name:
                    param.requires_grad_(False)

    def tokenize(self, vh_seqs, vl_seqs, device):
        paired = list(zip(vh_seqs, vl_seqs))
        return self.tokenizer(paired, pad=True, w_extra_tkns=True).to(device)

    def forward(self, tokens):
        reps = self.ablang.AbLang.AbRep(tokens)
        hidden = reps.last_hidden_states
        pad_mask = tokens.eq(self.padding_tkn).unsqueeze(-1)
        hidden = hidden.masked_fill(pad_mask, 0.0)
        lengths = (~pad_mask).sum(dim=1).clamp(min=1).float()
        return hidden.sum(dim=1) / lengths


class ApprovalHead(nn.Module):
    def __init__(self, hidden_size, dropout=HEAD_DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, 128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, emb):
        return self.net(emb).squeeze(-1)


class AntibodyDataset(torch.utils.data.Dataset):
    def __init__(self, vh, vl, labels):
        self.vh, self.vl, self.labels = vh, vl, labels
    def __len__(self):
        return len(self.vh)
    def __getitem__(self, i):
        return self.vh[i], self.vl[i], self.labels[i]

def collate_fn(batch):
    vh, vl, labels = zip(*batch)
    return list(vh), list(vl), torch.tensor(labels, dtype=torch.float32)

def train_epoch(encoder, head, loader, optimizer, loss_fn, device):
    encoder.train(); head.train()
    total = 0.0
    for vh, vl, labels in loader:
        tokens = encoder.tokenize(vh, vl, device)
        labels = labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(head(encoder(tokens)), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(list(encoder.parameters()) + list(head.parameters()), 1.0)
        optimizer.step()
        total += loss.item() * len(labels)
    return total / len(loader.dataset)

@torch.no_grad()
def predict(encoder, head, loader, device):
    encoder.eval(); head.eval()
    preds, labels = [], []
    for vh, vl, lb in loader:
        tokens = encoder.tokenize(vh, vl, device)
        preds.append(head(encoder(tokens)).cpu().numpy())
        labels.append(lb.numpy())
    return np.concatenate(preds), np.concatenate(labels)

In [ ]:
def run_cdr3_ablation_cv(df, use_cdr3_only: bool, device=DEVICE):
    """Run approval CV with optional CDR3 masking."""
    approval_df = df.dropna(subset=[LABEL_COL]).copy()
    folds = sorted(approval_df[FOLD_COL].unique())
    results = []
    tag = "CDR3-only" if use_cdr3_only else "Full-seq"

    for fold in folds:
        print(f"  [{tag}] Fold {fold}")
        train_df = approval_df[approval_df[FOLD_COL] != fold].reset_index(drop=True)
        test_df  = approval_df[approval_df[FOLD_COL] == fold].reset_index(drop=True)

        if use_cdr3_only:
            train_vh = [extract_cdr3_mask(s, "H") for s in train_df[VH_COL]]
            train_vl = [extract_cdr3_mask(s, "L") for s in train_df[VL_COL]]
            test_vh  = [extract_cdr3_mask(s, "H") for s in test_df[VH_COL]]
            test_vl  = [extract_cdr3_mask(s, "L") for s in test_df[VL_COL]]
        else:
            train_vh, train_vl = train_df[VH_COL].tolist(), train_df[VL_COL].tolist()
            test_vh,  test_vl  = test_df[VH_COL].tolist(),  test_df[VL_COL].tolist()

        y_train = train_df[LABEL_COL].values.astype(np.float32)
        y_test  = test_df[LABEL_COL].values.astype(np.float32)

        base = ablang2.pretrained(model_to_use="ablang2-paired", device=str(device))
        encoder = AbLang2LoRAEncoder(base).to(device)
        hidden_size = base.hparams.hidden_embed_size
        head = ApprovalHead(hidden_size).to(device)

        train_ds = AntibodyDataset(train_vh, train_vl, y_train)
        test_ds  = AntibodyDataset(test_vh, test_vl, np.zeros(len(test_df), dtype=np.float32))
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
        test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

        pos_w = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)]).to(device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
        opt = torch.optim.AdamW(
            [p for p in list(encoder.parameters()) + list(head.parameters()) if p.requires_grad],
            lr=LR, weight_decay=WEIGHT_DECAY)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR * 0.01)

        for epoch in range(EPOCHS):
            train_epoch(encoder, head, train_loader, opt, loss_fn, device)
            sched.step()

        preds, _ = predict(encoder, head, test_loader, device)
        probs = torch.sigmoid(torch.tensor(preds)).numpy()
        auprc = average_precision_score(y_test, probs)
        auroc = roc_auc_score(y_test, probs)
        print(f"    AUPRC={auprc:.3f} | AUROC={auroc:.3f}")
        results.append({"fold": fold, "condition": tag, "AUPRC": auprc, "AUROC": auroc})

    return pd.DataFrame(results)

In [ ]:
print("Running Full-sequence ablation...")
full_results = run_cdr3_ablation_cv(df_model, use_cdr3_only=False)

print("\nRunning CDR3-only ablation...")
cdr3_results = run_cdr3_ablation_cv(df_model, use_cdr3_only=True)

ablation_cdr3 = pd.concat([full_results, cdr3_results], ignore_index=True)
ablation_cdr3.to_csv(f"{OUTPUT_DIR}/ablation_cdr3_vs_fullseq.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

summary = (
    ablation_cdr3.groupby("condition")[["AUPRC", "AUROC"]]
    .agg(["mean", "std"]).round(3)
)
summary.columns = ["_".join(c) for c in summary.columns]
summary = summary.reset_index()

print("Ablation 2: CDR3-only vs Full-sequence")
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(2)
w = 0.3
ax.bar(x - w/2, summary["AUPRC_mean"], w, yerr=summary["AUPRC_std"],
       label="AUPRC", color="#2d6a9f", capsize=4)
ax.bar(x + w/2, summary["AUROC_mean"], w, yerr=summary["AUROC_std"],
       label="AUROC", color="#e07b39", capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(summary["condition"])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Ablation: CDR3-only vs. Full-sequence Input")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ablation_cdr3_vs_fullseq.png", dpi=150)
plt.show()

summary.to_csv(f"{OUTPUT_DIR}/ablation_cdr3_vs_fullseq_summary.csv", index=False)